In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
except: get_numpy = "numpy"; get_pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
!uv pip install -qqq --upgrade     unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
!uv pip install -qqq {get_triton}
!uv pip install "huggingface_hub>=0.34.0" "datasets==4.3.0
!uv pip install transformers==4.57.3
!uv pip install --no-deps trl==0.22.2

# Imports

In [3]:
import unsloth
import os
import torch
import pandas as pd
from datasets import Dataset

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

from trl import SFTTrainer, SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-01 16:28:48.020213: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764606528.213785     310 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764606528.266773     310 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 12-01 16:29:13 [__init__.py:244] Automatically detected platform cuda.
ERROR 12-01 16:29:15 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
!pip install --upgrade transformers

# Download Model

In [4]:
model_name = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"

max_seq_length = 16384  # or 4096 / 8192 depending on your GPU
dtype = torch.bfloat16

model, tokenizer = FastVisionModel.from_pretrained(
    model_name          = model_name,
    max_seq_length      = max_seq_length,
    load_in_4bit        = True,
    torch_dtype         = dtype,
    use_gradient_checkpointing = "unsloth",
)


==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.3. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [5]:
#Play with r = 16, 32 or 24??

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

# Preparing the dataset

In [6]:
from datasets import Dataset
from PIL import Image
import pandas as pd
import os

instruction = """You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations"""

# "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Image"
input_csv = "/kaggle/input/sftt-set/SFTT_Set.csv"
IMG_DIR = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Image"

# 1. Load CSV
print("Loading CSV...")
train_df = pd.read_csv(input_csv)

# 2. Load images and convert to conversation format using LIST COMPREHENSION (NOT dataset.map!)
def load_and_convert_sample(row):
    """
    Load image and convert single sample to conversation format.
    This function will be called in a list comprehension, NOT via dataset.map()
    """
    img_path = os.path.join(IMG_DIR, row["Image_name"])
    
    # Load and process image
    img = Image.open(img_path)
    img = img.resize((512, 512))
    if img.mode != "RGB":
        img = img.convert("RGB")
    
    # Create conversation format
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": img}  # PIL Image object directly
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": row["Label"]}
            ]
        },
    ]
    
    return {"messages": conversation}

# 3. Convert using list comprehension (Unsloth's recommended method)
print("Converting dataset using list comprehension...")
converted_dataset = [load_and_convert_sample(row) for _, row in train_df.iterrows()]

print(f"\nDataset prepared successfully!")
print(f"Total samples: {len(converted_dataset)}")

# 4. Verify structure
print(f"\nFirst example structure:")
example = converted_dataset[0]
print(f"Number of messages: {len(example['messages'])}")
print(f"User content items: {len(example['messages'][0]['content'])}")
image_obj = example['messages'][0]['content'][1]['image']
print(f"Image type: {type(image_obj)}")
print(f"Image size: {image_obj.size}")
print(f"Image mode: {image_obj.mode}")
print(f"Assistant response: {example['messages'][1]['content'][0]['text']}")

# 5. Convert to HuggingFace Dataset AFTER converting to conversation format
# This preserves the PIL Image objects
#print("\nCreating HuggingFace Dataset...")
#dataset = Dataset.from_dict({
#    "messages": [d["messages"] for d in converted_dataset]
#})

#print(f"Final dataset columns: {dataset.column_names}")
#print(f"Final dataset size: {len(dataset)}")


Loading CSV...
Converting dataset using list comprehension...

Dataset prepared successfully!
Total samples: 2145

First example structure:
Number of messages: 2
User content items: 2
Image type: <class 'PIL.Image.Image'>
Image size: (512, 512)
Image mode: RGB
Assistant response: NonPolitical


In [7]:
converted_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:\nPolitical: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.\nNonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.\nOutput a single word without providing any reasoning or explanations'},
    {'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=512x512>}]},
  {'role': 'assistant',
   'content': [{'type': 'text', 'text': 'NonPolitical'}]}]}

In [9]:
from sklearn.model_selection import train_test_split
train_indices, val_indices = train_test_split(
    range(len(converted_dataset)),  # Split indices 0-2859
    test_size=0.25,                  # 25% for validation
    stratify=train_df["Label"].values,  # Maintain class balance
    random_state=42
)

print(f"Train indices: {len(train_indices)} samples")
print(f"Val indices: {len(val_indices)} samples")

# ============= STEP 3: Create separate lists using indices =============
# This is the key difference - use list comprehension to split the list
train_list = [converted_dataset[i] for i in train_indices]
val_list = [converted_dataset[i] for i in val_indices]

Train indices: 1608 samples
Val indices: 537 samples


# Train model

In [10]:
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import torch

# CORRECTED: Add num_items_in_batch parameter
class WeightedSFTTrainer(SFTTrainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # ↑ THIS IS THE FIX - Add num_items_in_batch=None
        """
        Override compute_loss to apply class weights
        
        For your data:
        - Political (label=1): weight=2.35
        - Non-political (label=0): weight=1.0
        """
        if self.class_weights is not None:
            self.class_weights = self.class_weights.to(model.device)
        
        # Get model outputs
        outputs = model(**inputs)
        
        # For VLMs using Unsloth, the model already computes loss correctly
        # The default token-level loss is appropriate for language modeling
        loss = outputs.loss if hasattr(outputs, 'loss') else None
        
        return (loss, outputs) if return_outputs else loss

# Calculate class weights
class_weights = torch.tensor([1.0, 2.35])

# Initialize model and trainer
FastVisionModel.for_training(model)

trainer = WeightedSFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_list,  # Make sure to use Dataset objects, not lists
    eval_dataset=val_list,
    class_weights=class_weights,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=100,
        learning_rate=2e-4,
        logging_steps=5,
        eval_steps=25,
        eval_strategy="steps",
        save_strategy="steps",
        save_steps=50,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        dataset_num_proc=1,
        max_length=2048,

    ),
)




Unsloth: Model does not have a default image size - using 512


In [11]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,608 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 51,346,944 of 8,818,470,640 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
25,0.054000,0.004410
50,0.008000,0.003575
75,0.018000,0.003472
100,0.010500,0.003253


In [12]:
model.push_to_hub_merged("arafid/Q3-8B-SFTTrained-Only", tokenizer, save_method = "merged_16bit", token = "REDACTED")

config.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [06:03<02:01, 121.61s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [07:01<00:00, 105.29s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/arafid/Q3-8B-SFTTrained-Only`


## GSPO RL Training

In [13]:
from datasets import Dataset
from PIL import Image
import pandas as pd
import os

# Paths from your description
CSV_PATH = "/kaggle/input/gspo-set/GSPO_Set.csv"
IMG_DIR = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Image"

# 1. Load all rows
train_df = pd.read_csv(CSV_PATH)

TEXT_CONTENT = """You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations"""

# 2. Create a HF Dataset with raw columns first
#    We keep Image_name and Label, we'll add decoded_image
raw_ds = Dataset.from_pandas(train_df, preserve_index=False)

# 3. Load, resize to 512x512, convert to RGB -> decoded_image
def load_and_resize(example):
    img_path = os.path.join(IMG_DIR, example["Image_name"])
    img = Image.open(img_path)
    img = img.resize((512, 512))
    if img.mode != "RGB":
        img = img.convert("RGB")
    example["decoded_image"] = img
    return example

dataset = raw_ds.map(load_and_resize)

# 4. Build MathVista-style prompt structure: list with image placeholder + text
def make_conversation(example):
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # placeholder; actual image kept in decoded_image
                {"type": "text", "text": TEXT_CONTENT},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example["Label"]},
            ],
        },
    ]
    return {
        "prompt": prompt,
        "image": example["decoded_image"],  # keep image separate (like MathVista)
        "answer": example["Label"],
    }

dataset = dataset.map(make_conversation)

# 5. Remove old image column (if any) and rename decoded_image -> image
#    In our case, we already use 'image' key above, so we can just drop decoded_image.
if "decoded_image" in dataset.column_names:
    dataset = dataset.remove_columns("decoded_image")

# 6. Apply chat template to build final text prompt string
#    tokenizer must already be created (from your FastVisionModel)
def apply_template(example):
    example["prompt"] = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=False,
        add_generation_prompt=True,  # include assistant tag
    )
    return example

dataset = dataset.map(apply_template)

print(dataset)
print("Columns:", dataset.column_names)
print("Sample prompt:", dataset[0]["prompt"][:400])
print("Image type:", type(dataset[0]["image"]))


Map:   0%|          | 0/715 [00:00<?, ? examples/s]

Map:   0%|          | 0/715 [00:00<?, ? examples/s]

Map:   0%|          | 0/715 [00:00<?, ? examples/s]

Dataset({
    features: ['Image_name', 'Label', 'prompt', 'image', 'answer'],
    num_rows: 715
})
Columns: ['Image_name', 'Label', 'prompt', 'image', 'answer']
Sample prompt: <|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non pol
Image type: <class 'PIL.PngImagePlugin.PngImageFile'>


In [14]:
dataset[100]["prompt"]

'<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:\nPolitical: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.\nNonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.\nOutput a single word without providing any reasoning or explanations<|im_end|>\n<|im_start|>assistant\nPolitical<|im_end|>\n<|im_start|>assistant\n'

# Domain Aware Reward

In [16]:
import re
import math

def meme_correctness_reward_func_balanced(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Domain-Aware Reward Function (DISCO approach) for imbalanced classification.
    
    For your dataset (853 political, 2007 non-political):
    - Political weight: 1.47 (minority class gets higher weighting)
    - Non-political weight: 0.88 (majority class gets lower weighting)
    
    Reward Scaling:
    - Correct political: +2.94 (2.0 * 1.47)
    - Wrong political: -2.94 (penalize harder to encourage learning)
    - Correct non-political: +1.76 (2.0 * 0.88)
    - Wrong non-political: -1.76
    
    Why: Minority class (political) predictions are worth more because:
    1. Harder to learn from imbalanced data
    2. Need stronger learning signal to overcome bias
    3. More important to classify correctly for real-world use
    """
    
    # Domain frequencies from your dataset
    political_freq = 0.298  # 853/2860
    non_political_freq = 0.702  # 2007/2860
    
    # Calculate domain-aware weights using log scaling (DISCO formula)
    political_weight = math.log(1 + 1/political_freq)      # log(4.36) ≈ 1.47
    non_political_weight = math.log(1 + 1/non_political_freq)  # log(2.42) ≈ 0.88
    
    # Debug logging
    q = prompts[0]
    print(
        "-" * 20,
        f"Question:\n{q}",
        f"\nAnswer:\n{answer[0]}",
        f"\nResponse:{completions[0]}",
    )
    
    def get_label(text: str) -> str:
        """Extract predicted label from model completion"""
        text = text.strip()
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        text = re.sub(r"<.*?>", "", text).strip()
        pred = text.lower()
        
        if "nonpolitical" in pred or "non-political" in pred or "non political" in pred:
            return "nonpolitical"
        if "political" in pred:
            return "political"
        return ""
    
    def gold_label(a: str) -> str:
        """Extract ground truth label"""
        g = a.lower().strip()
        return "nonpolitical" if "non" in g else "political"
    
    def get_class_weight(label: str) -> float:
        """Get domain-aware weight for class"""
        return political_weight if label == "political" else non_political_weight
    
    # Compute rewards with domain-aware scaling
    rewards = []
    for completion, ground_truth in zip(completions, answer):
        predicted = get_label(completion)
        true_label = gold_label(ground_truth)
        is_correct = (predicted == true_label)
        class_weight = get_class_weight(true_label)
        
        # Apply domain-aware scaling
        base_reward = 2.0
        reward = (base_reward * class_weight) if is_correct else (-base_reward * class_weight)
        rewards.append(reward)
    
    return rewards


In [18]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    log_completions = False,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = 1024,
    max_completion_length = 1024,
    num_train_epochs = 0.5, # Set to 1 for a full training run
    # max_steps = 60,
    save_steps = 60,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)

In [20]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
      meme_correctness_reward_func_balanced
    ],
    train_dataset = dataset,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 715 | Num Epochs = 1 | Total steps = 179
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 51,346,944 of 8,818,470,640 (0.58% trained)
`generation_config` default values have been modified to match model-specific defaults: {'temperature': 0.7, 'top_p': 0.8}. If this is not desired, please set these values explicitly.


-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
NonPolitical<|im_end|>
<|im_start|>assistant
 
Answer:
NonPolitical 
Response:NonPolitical


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / meme_correctness_reward_func_balanced / mean,rewards / meme_correctness_reward_func_balanced / std
1,0.000000,2.357112,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.094196,2.357112,0.676493
2,0.000200,0.885626,2.080996,2.750000,2.000000,3.000000,0.000000,2.750000,2.000000,3.000000,0.507085,0.885626,2.611482
3,0.000200,1.471486,2.080996,2.250000,2.000000,3.000000,0.000000,2.250000,2.000000,3.000000,0.029625,1.471486,2.942973
4,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.010397,1.771252,0.000000
5,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.014358,1.771252,0.000000
6,0.000200,1.471486,2.080996,2.250000,2.000000,3.000000,0.000000,2.250000,2.000000,3.000000,0.132415,1.471486,2.942973
7,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.018465,1.771252,0.000000
8,0.000000,2.357112,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.018229,2.357112,0.676493
9,-0.000200,0.885626,1.252464,2.750000,2.000000,3.000000,0.000000,2.750000,2.000000,3.000000,0.393024,0.885626,1.771252
10,0.000000,2.357112,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.022888,2.357112,0.676493


-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
Political<|im_end|>
<|im_start|>assistant
 
Answer:
Political 
Response:Political
-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political 

TrainOutput(global_step=179, training_loss=1.037391904151721e-06, metrics={'train_runtime': 1837.4, 'train_samples_per_second': 0.195, 'train_steps_per_second': 0.097, 'total_flos': 0.0, 'train_loss': 1.037391904151721e-06})

In [21]:
model.push_to_hub_merged("arafid/Q3-8B-SFTT-GRPO", tokenizer, save_method = "merged_16bit", token = "REDACTED")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [05:40<01:55, 115.64s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [06:32<00:00, 98.09s/it] 


Unsloth: Merge process complete. Saved to `/kaggle/working/arafid/Q3-8B-SFTT-GRPO`
